# Comparativo 2020-2025 — IA Embeddings
Roda o pipeline da Opção 2 para todos os anos e plota evolução do enquadramento.


In [ ]:
from pathlib import Path
import sys
sys.path.append(str(Path.cwd()))
from config import *
from utils import extract_text, chunk_por_paragrafo, filtrar_chunks_com_termo, carregar_modelo, gerar_centroides, embed_chunks, classificar_chunks
from collections import Counter
try:
    import pandas as pd
    HAS_PANDAS=True
except ImportError:
    HAS_PANDAS=False
    print("pandas não instalado — usando fallback sem pandas (pip install pandas)")

PDF_DIR=Path("/workspaces/governanca-digital_mre/relatorios-gestao-mre")
try:
    model=carregar_modelo(MODELO_EMBEDDING)
    centroide_A, centroide_B = gerar_centroides(model, ABORDAGEM_A_TERMOS, ABORDAGEM_B_TERMOS, ABORDAGEM_A_FRASES, ABORDAGEM_B_FRASES)
    modelo_ok=True
except Exception as e:
    print(f"Modelo não carregado: {e}")
    print("Instale: pip install -r requirements.txt")
    modelo_ok=False
    # sem modelo, não dá para continuar
rows=[]
if modelo_ok:
    for ano, pdf_file in sorted(MAPA_ARQUIVOS.items()):
        txt=extract_text(PDF_DIR/pdf_file)
        chunks=chunk_por_paragrafo(txt)
        filtrados=filtrar_chunks_com_termo(chunks, TERMO_CENTRAL_REGEX)
        if not filtrados:
            rows.append({"ano":ano,"chunks":0,"A":0,"B":0,"neutro":0,"media_diff":0})
            continue
        embs=embed_chunks(model, [f['texto'] for f in filtrados])
        res=classificar_chunks(embs, centroide_A, centroide_B)
        cnt=Counter(r['vencedor'] for r in res)
        import numpy as np
        rows.append({"ano":ano,"chunks":len(filtrados),"A":cnt.get('A',0),"B":cnt.get('B',0),"neutro":cnt.get('neutro',0),"media_diff":float(np.mean([r['diff'] for r in res]))})
    if HAS_PANDAS:
        df=pd.DataFrame(rows)
        display(df)
    else:
        for r in rows:
            print(r)
        df=None
else:
    print("Sem modelo, pulando agregação")
    rows=[]
    df=None


In [ ]:
try:
    import matplotlib.pyplot as plt
    HAS_MPL=True
except ImportError:
    HAS_MPL=False
    print("matplotlib não instalado — pip install matplotlib")
if 'rows' in locals() and rows and HAS_MPL:
    if 'df' in locals() and df is not None:
        df_plot=df.set_index('ano')[['A','B','neutro']]
        ax=df_plot.plot(kind='bar', stacked=True, figsize=(8,4), color=['purple','orange','gray'])
        plt.title("Evolução: chunks com 'internet' por enquadramento (IA)")
        plt.ylabel("n chunks")
        plt.xticks(rotation=0)
        plt.legend(title="vencedor")
        plt.tight_layout()
        plt.show()
        plt.figure(figsize=(7,3))
        plt.plot(df['ano'], df['media_diff'], marker='o')
        plt.axhline(0,color='gray',ls='--')
        plt.title("Média diff (A-B): >0 = soberano, <0 = mercado")
        plt.grid(alpha=0.3)
        plt.show()
    else:
        # fallback sem pandas
        anos=[r['ano'] for r in rows]
        diffs=[r['media_diff'] for r in rows]
        plt.figure(figsize=(7,3))
        plt.plot(anos, diffs, marker='o')
        plt.axhline(0,color='gray',ls='--')
        plt.title("Média diff (A-B)")
        plt.show()
else:
    if not HAS_MPL:
        pass
    else:
        print("Sem dados para plotar")


In [ ]:
try:
    if 'df' in locals() and df is not None:
        import pandas as pd
        df.to_csv("comparativo_ia.csv", index=False)
        print("Salvo comparativo_ia.csv via pandas")
    else:
        raise ImportError
except Exception:
    import csv
    if 'rows' in locals() and rows:
        keys=list(rows[0].keys())
        with open("comparativo_ia.csv","w",newline='',encoding='utf-8') as f:
            w=csv.DictWriter(f, fieldnames=keys)
            w.writeheader()
            w.writerows(rows)
        print("Salvo comparativo_ia.csv (fallback)")
